# AVG Testing v5 — official test tanpa fitting


## 1. Muat dan verifikasi artefak final


In [ ]:
from pathlib import Path
import gc, hashlib, json, os, time
import joblib
import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt

SEED = 30092026
RUN_ID = "cifar10_7fitur_v5"
VARIANT = "avg"
ROOT = Path.cwd().resolve()
OUT = ROOT / "outputs" / RUN_ID / VARIANT
MODEL_PATH = OUT / "model" / "cnn.keras"
SVM_PATH = OUT / "model" / "svm.joblib"
FEATURE_PATH = OUT / "fitur" / "train_features.npz"
REPORT_PATH = OUT / "laporan" / "cnn_training.json"
CLASS_NAMES = (
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
)


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for part in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(part)
    return h.hexdigest()


for path in (MODEL_PATH, SVM_PATH, FEATURE_PATH, REPORT_PATH):
    if not path.is_file():
        raise FileNotFoundError(
            "Artefak Training/Extraction tidak lengkap: " + str(path)
        )
report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
svm = joblib.load(SVM_PATH)
with np.load(FEATURE_PATH, allow_pickle=False) as archive:
    if (
        str(archive["cnn_sha256"]) != report["cnn_sha256"]
        or str(archive["split_hash"]) != report["split_hash"]
        or str(archive["run_id"]) != RUN_ID
    ):
        raise ValueError("Arsip training tidak cocok")
if (
    sha256_file(MODEL_PATH) != report["cnn_sha256"]
    or svm["cnn_sha256"] != report["cnn_sha256"]
    or svm["fit_count"] != 50000
    or svm["representation"] != "B"
    or svm["run_id"] != RUN_ID
):
    raise ValueError("CNN/SVM final tidak kompatibel")
print("Artefak cocok:", VARIANT)


## 2. Muat official test CIFAR-10


In [ ]:
from tensorflow.keras.datasets import cifar10

(_, _), (x_test, y_test) = cifar10.load_data()
y_test = y_test.ravel().astype(np.int64)
if x_test.shape != (10000, 32, 32, 3) or y_test.shape != (10000,):
    raise ValueError("Official test CIFAR-10 tidak sesuai")
print("Official test:", x_test.shape)


## 3. Konversi input identik dengan Training


In [ ]:
def prepare_images(x, variant):
    rgb = np.asarray(x, dtype=np.float32) / np.float32(255.0)
    if variant == "rgb":
        return rgb
    if variant == "avg":
        return np.mean(rgb, axis=-1, keepdims=True, dtype=np.float32)
    if variant == "ntsc":
        weights = np.array([0.299, 0.587, 0.114], dtype=np.float32)
        return np.sum(rgb * weights, axis=-1, keepdims=True, dtype=np.float32)
    raise ValueError("Varian tidak dikenal: " + variant)


## 4. Ekstraksi embedding test tanpa update bobot


In [ ]:
cnn = tf.keras.models.load_model(MODEL_PATH, compile=False)
extractor = tf.keras.Model(cnn.input, cnn.get_layer("embedding").output)
classifier_and_embedding = tf.keras.Model(
    cnn.input, [cnn.get_layer("embedding").output, cnn.output]
)
X_test = np.empty((10000, 512), np.float32)
cnn_pred = np.empty(10000, np.int64)
for begin in range(0, 10000, 128):
    end = min(begin + 128, 10000)
    x = prepare_images(x_test[begin:end], VARIANT)
    original, cnn_scores = classifier_and_embedding(x, training=False)
    flipped = extractor(np.ascontiguousarray(x[:, :, ::-1, :]), training=False)
    X_test[begin:end] = (
        original.numpy().astype(np.float32) + flipped.numpy().astype(np.float32)
    ) * np.float32(0.5)
    cnn_pred[begin:end] = np.argmax(cnn_scores.numpy(), axis=1)
    if end % 2560 == 0 or end == 10000:
        print("Testing", VARIANT, end, "/10000", flush=True)
if not np.isfinite(X_test).all():
    raise ValueError("Fitur test tidak valid")
del extractor, classifier_and_embedding, cnn
tf.keras.backend.clear_session()
gc.collect()


## 5. Prediksi SVM yang sudah dibekukan


In [ ]:
pred = svm["pipeline"].predict(X_test)
if pred.shape != (10000,) or not np.isin(pred, np.arange(10)).all():
    raise ValueError("Prediksi SVM tidak valid")
print("Prediksi selesai:", len(pred))


## 6. Hitung accuracy dan metrik per kelas


In [ ]:
cnn_accuracy = float(accuracy_score(y_test, cnn_pred))
cnn_correct = int(np.sum(y_test == cnn_pred))
accuracy = float(accuracy_score(y_test, pred))
macro_f1 = float(f1_score(y_test, pred, average="macro"))
correct = int(np.sum(y_test == pred))
report_dict = classification_report(
    y_test, pred, target_names=CLASS_NAMES, output_dict=True
)
cm = confusion_matrix(y_test, pred)
print(classification_report(y_test, pred, target_names=CLASS_NAMES))
print(f"{'Model':<20} {'Benar / 10000':>15} {'Akurasi':>12}")
print("-" * 49)
print(f"{'CNN softmax':<20} {cnn_correct:>15} {cnn_accuracy:>11.2%}")
print(f"{'CNN + SVM':<20} {correct:>15} {accuracy:>11.2%}")
print("Post-exposure testing: official test telah dipakai pada eksperimen sebelumnya")


## 7. Simpan metrik dan prediksi per sampel


In [ ]:
metrics_path = OUT / "laporan" / "svm_test_v5.json"
predictions_path = OUT / "laporan" / "svm_test_predictions_v5.npz"
metadata = {
    "run_id": RUN_ID,
    "variant": VARIANT,
    "cnn_sha256": report["cnn_sha256"],
    "svm_sha256": sha256_file(SVM_PATH),
    "feature_archive_sha256": sha256_file(FEATURE_PATH),
    "accuracy": accuracy,
    "macro_f1": macro_f1,
    "correct": correct,
    "total": 10000,
    "cnn_accuracy": cnn_accuracy,
    "cnn_correct": cnn_correct,
    "post_exposure_testing": True,
    "classification_report": report_dict,
    "confusion_matrix": cm.tolist(),
}
if metrics_path.exists():
    old = json.loads(metrics_path.read_text(encoding="utf-8"))
    if old["svm_sha256"] != metadata["svm_sha256"]:
        raise ValueError("Laporan test lama memakai model berbeda")
temporary = metrics_path.with_suffix(".json.tmp")
temporary.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)
os.replace(temporary, metrics_path)
if predictions_path.exists():
    with np.load(predictions_path, allow_pickle=False) as old:
        if str(old["svm_sha256"]) != metadata["svm_sha256"]:
            raise ValueError("Prediksi test lama memakai model berbeda")
with predictions_path.open("wb") as stream:
    np.savez_compressed(
        stream,
        indices=np.arange(10000),
        y_true=y_test,
        y_pred=pred,
        y_pred_cnn=cnn_pred,
        svm_sha256=metadata["svm_sha256"],
    )
print("Laporan:", metrics_path)


## 8. Visualisasi confusion matrix


In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
    ax=ax, xticks_rotation=45, colorbar=False
)
fig.tight_layout()
plot_path = OUT / "laporan" / "svm_confusion_v5.png"
fig.savefig(plot_path, dpi=150)
plt.show()
print("Grafik:", plot_path)
